# LAB 1
## Implementation and Performance Analysis of Q-Learning Using an Epsilon-Greedy Policy

### Aim
Implement Q-learning in a Gymnasium environment and study how training length affects cumulative reward and learning performance.

This notebook uses deterministic 'FrozenLake-v1' so the learning trend is easy to observe.

### Tasks covered
Train with 500, 1,000, 2,000, 5,000, and 10,000 episodes; decay epsilon; record rewards; plot cumulative and latest-100 average rewards; test each policy for 100 episodes without exploration; compare success rates; and visualize the environment and extracted policy.

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

EPISODE_CONFIGS = [500, 1_000, 2_000, 5_000, 10_000]
TEST_EPISODES, MAX_STEPS, SEED = 100, 100, 42
LEARNING_RATE, DISCOUNT_FACTOR = 0.8, 0.95
EPSILON_START, EPSILON_MIN, EPSILON_DECAY = 1.0, 0.01, 0.001

In [ ]:
def train_q_learning(number_of_episodes, seed=SEED):
    env = gym.make("FrozenLake-v1", is_slippery=False)
    env.action_space.seed(seed)
    rng = np.random.default_rng(seed)
    q_table = np.zeros((env.observation_space.n, env.action_space.n))
    rewards, epsilon = [], EPSILON_START
    for episode in range(number_of_episodes):
        state, _ = env.reset(seed=seed + episode)
        total_reward = 0
        for _ in range(MAX_STEPS):
            action = env.action_space.sample() if rng.random() < epsilon else int(np.argmax(q_table[state]))
            next_state, reward, terminated, truncated, _ = env.step(action)
            target = reward if terminated else reward + DISCOUNT_FACTOR * np.max(q_table[next_state])
            q_table[state, action] += LEARNING_RATE * (target - q_table[state, action])
            state, total_reward = next_state, total_reward + reward
            if terminated or truncated: break
        rewards.append(total_reward)
        epsilon = max(EPSILON_MIN, epsilon * np.exp(-EPSILON_DECAY))
    env.close()
    return q_table, np.array(rewards)

def evaluate_policy(q_table, number_of_episodes=TEST_EPISODES, seed=SEED):
    env, successes = gym.make("FrozenLake-v1", is_slippery=False), 0
    for episode in range(number_of_episodes):
        state, _ = env.reset(seed=seed + episode)
        for _ in range(MAX_STEPS):
            state, reward, terminated, truncated, _ = env.step(int(np.argmax(q_table[state])))
            if reward == 1: successes += 1
            if terminated or truncated: break
    env.close()
    return successes / number_of_episodes * 100

In [ ]:
results, summary = {}, []
for episodes in EPISODE_CONFIGS:
    q_table, rewards = train_q_learning(episodes)
    results[episodes] = {"q_table": q_table, "rewards": rewards}
    summary.append({"Training episodes": episodes,
                    "Final cumulative reward": rewards.sum(),
                    "Average reward (last 100)": rewards[-100:].mean(),
                    "Test success rate (%)": evaluate_policy(q_table)})
summary_df = pd.DataFrame(summary)
summary_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for episodes, result in results.items():
    rewards = result["rewards"]
    axes[0].plot(np.cumsum(rewards), label=f"{episodes:,} episodes")
    axes[1].plot(pd.Series(rewards).rolling(100, min_periods=1).mean(), label=f"{episodes:,} episodes")
axes[0].set(title="Cumulative reward", xlabel="Training episode", ylabel="Cumulative reward")
axes[1].set(title="Average reward over latest 100 episodes", xlabel="Training episode", ylabel="Average reward")
for ax in axes: ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.show()

plt.figure(figsize=(8, 4))
plt.bar(summary_df["Training episodes"].astype(str), summary_df["Test success rate (%)"])
plt.title("Policy success rate after training"); plt.xlabel("Training episodes"); plt.ylabel("Success rate (%)"); plt.ylim(0, 105); plt.grid(axis="y", alpha=.3); plt.show()

## Environment and extracted policy
The arrows show the greedy action selected by the best-performing Q-table. 'S' is the start, 'H' is a hole, and 'G' is the goal.

In [ ]:
best_episodes = summary_df.loc[summary_df["Test success rate (%)"].idxmax(), "Training episodes"]
policy = np.argmax(results[best_episodes]["q_table"], axis=1).reshape(4, 4)
symbols, lake = {0:"←", 1:"↓", 2:"→", 3:"↑"}, np.array([list(x) for x in ["SFFF", "FHFH", "FFFH", "HFFG"]])
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="rgb_array"); env.reset(seed=SEED)
axes[0].imshow(env.render()); axes[0].set_title("FrozenLake environment"); axes[0].axis("off"); env.close()
axes[1].imshow(np.zeros((4, 4)), cmap="Blues", vmin=0, vmax=1)
for row in range(4):
    for col in range(4):
        label = lake[row, col] if lake[row, col] in "SHG" else symbols[policy[row, col]]
        axes[1].text(col, row, label, ha="center", va="center", fontsize=20)
axes[1].set(xticks=range(4), yticks=range(4), xticklabels=[], yticklabels=[], title=f"Greedy policy ({best_episodes:,} episodes)")
axes[1].grid(color="white", linewidth=2); plt.tight_layout(); plt.show()
print("Policy action numbers (0=left, 1=down, 2=right, 3=up):"); print(policy)

## Conclusion
The table and plots compare how training length changes cumulative reward, recent average reward, and greedy-policy success rate. In this deterministic environment, sufficient training should learn a route from 'S' to 'G'.